<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 3: Süpermarket Satış Görselleştirme

**VERİ BİLİMİ TEMELLERİ** · Modül 3 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta03/hafta03_supermarket_gorsellestirme.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta03/hafta03_supermarket_gorsellestirme.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta03_veri_gorsellestirme.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/03/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - Süpermarket satış verileri analizi
> - Çoklu grafik türleri ile görselleştirme
> - Kategori bazlı satış karşılaştırması

# Hafta 3: Süpermarket Satış Verisi Görselleştirme

Bu defterde bir süpermarket satış veri setini çeşitli grafiklerle analiz edip görselleştireceğiz.

## İçindekiler
1. Veri yükleme ve inceleme
2. Ürün kategorisi satış çubuk grafiği
3. Aylık satış trendi çizgi grafiği
4. Ödeme yöntemi dağılımı pasta grafiği
5. Cinsiyete dayalı analiz
6. Şube karşılaştırma kutu grafiği
7. Korelasyon ısı haritası

## 1. Kütüphaneler ve Veri Yükleme

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")

print("Kütüphaneler yüklendi!")

## 2. Gerçek Süpermarket Satış Verisini Yükleme

Kaggle'daki **Supermarket Sales** veri setini kullanacağız. Bu veri seti Myanmar'daki 3 farklı süpermarket şubesinin 3 aylık gerçek satış verilerini içerir.

**Kaynak:** [Kaggle - Supermarket Sales](https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales)

**Sütunlar:** Invoice ID, Branch, City, Customer type, Gender, Product line, Unit price, Quantity, Tax 5%, Total, Date, Time, Payment, cogs, gross margin percentage, gross income, Rating

In [ ]:
# Kaggle Supermarket Sales veri setini yükle
# Yöntem 1: Doğrudan URL'den
try:
    url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/supermarket_sales.csv"
    df = pd.read_csv(url)
    print("Veri seti URL'den başarıyla yüklendi!")
except:
    # Yöntem 2: Kaggle'dan indirip yükleme
    print("URL çalışmıyor. Lütfen veri setini Kaggle'dan indirin:")
    print("https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales")
    print("\nGoogle Colab'da dosya yükleme:")
    print("from google.colab import files")
    print("uploaded = files.upload()")

# Tarih sütununu datetime'a çevir
df['Date'] = pd.to_datetime(df['Date'])

# Sütun isimlerini Türkçeleştirelim (orijinal isimleri de tutalım)
print(f"\nVeri seti boyutu: {df.shape}")
print(f"\nSütunlar: {list(df.columns)}")
print(f"\nŞubeler: {df['Branch'].unique()}")
print(f"Ürün grupları: {df['Product line'].unique()}")
df.head()

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
# Veri seti özet istatistikleri
print("Temel İstatistikler:")
print("=" * 50)
print(df.describe().round(2))
print("\nKategorik Değişkenler:")
print("=" * 50)
for col in ['Branch', 'Gender', 'Customer type', 'Payment', 'Product line']:
    print(f"\n{col}:")
    print(df[col].value_counts())

## 2. Ürün Kategorisi Satış Çubuk Grafiği

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Kategoriye göre toplam satış
kategori_satis = df.groupby('Product line')['gross income'].sum().sort_values(ascending=True)

plt.figure(figsize=(12, 6))
renkler = sns.color_palette('viridis', len(kategori_satis))
bars = plt.barh(kategori_satis.index, kategori_satis.values, color=renkler, edgecolor='black', linewidth=0.5)

# Değerleri çubukların yanına yaz
for bar, val in zip(bars, kategori_satis.values):
    plt.text(val + 500, bar.get_y() + bar.get_height()/2,
             f'{val:,.0f} ₺', va='center', fontsize=11, fontweight='bold')

plt.title('Ürün Kategorisine Göre Toplam Satış', fontsize=16, fontweight='bold')
plt.xlabel('Toplam Satış (₺)', fontsize=12)
plt.ylabel('Product line', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Aylık Satış Trendi Çizgi Grafiği

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Günlük toplam satış trendi
gunluk_satis = df.groupby(df['Date'].dt.date)['gross income'].sum().reset_index()
gunluk_satis.columns = ['Date', 'Toplam Satış']
gunluk_satis['Date'] = pd.to_datetime(gunluk_satis['Date'])

# 7 günlük hareketli ortalama
gunluk_satis['Hareketli Ort.'] = gunluk_satis['Toplam Satış'].rolling(window=7).mean()

plt.figure(figsize=(14, 6))
plt.plot(gunluk_satis['Date'], gunluk_satis['Toplam Satış'], alpha=0.4, color='steelblue', label='Günlük Satış')
plt.plot(gunluk_satis['Date'], gunluk_satis['Hareketli Ort.'], color='red', linewidth=2.5, label='7 Günlük Hareketli Ortalama')
plt.fill_between(gunluk_satis['Date'], gunluk_satis['Toplam Satış'], alpha=0.1, color='steelblue')

plt.title('Günlük Satış Trendi (Ocak - Mart 2024)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Toplam Satış (₺)', fontsize=12)
plt.legend(fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Aylık toplam satış
aylik_satis = df.groupby('Ay_No')['gross income'].sum()
ay_isimleri = ['Ocak', 'Şubat', 'Mart']

plt.figure(figsize=(10, 6))
plt.plot(ay_isimleri, aylik_satis.values, marker='o', linewidth=3, markersize=12,
         color='#2196F3', markerfacecolor='white', markeredgewidth=2)

for i, val in enumerate(aylik_satis.values):
    plt.annotate(f'{val:,.0f} ₺', (ay_isimleri[i], val),
                textcoords="offset points", xytext=(0, 15), ha='center', fontsize=12, fontweight='bold')

plt.title('Aylık Toplam Satış', fontsize=16, fontweight='bold')
plt.xlabel('Ay', fontsize=12)
plt.ylabel('Toplam Satış (₺)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Ödeme Yöntemi Dağılımı Pasta Grafiği

### Pasta Grafiği

Kategorilerin genel dağılım içindeki oranlarını pasta grafiği ile görselleştiriyoruz.

In [ ]:
# Ödeme yöntemi dağılımı
odeme_dagilimi = df['Payment'].value_counts()

renkler = ['#FF6B6B', '#4ECDC4', '#45B7D1']
patlama = [0.05, 0.05, 0.05]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Pasta grafiği
axes[0].pie(odeme_dagilimi.values, labels=odeme_dagilimi.index, autopct='%1.1f%%',
            colors=renkler, explode=patlama, shadow=True, startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('Ödeme Yöntemi Dağılımı (Adet)', fontsize=14, fontweight='bold')

# Ödeme yöntemine göre toplam satış
odeme_satis = df.groupby('Payment')['gross income'].sum()
axes[1].pie(odeme_satis.values, labels=odeme_satis.index, autopct='%1.1f%%',
            colors=renkler, explode=patlama, shadow=True, startangle=90,
            textprops={'fontsize': 13})
axes[1].set_title('Ödeme Yöntemi Dağılımı (Tutar)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Cinsiyete Dayalı Analiz

### Kutu Grafiği (Box Plot)

Kutu grafiği verinin dağılımını, medyanını, çeyrekliklerini ve uç değerlerini (outlier) gösterir. Gruplar arası karşılaştırma için idealdir.

In [ ]:
# Cinsiyet ve kategori bazlı analiz
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Cinsiyete göre kategori satışları
cinsiyet_kategori = df.groupby(['Product line', 'Gender'])['gross income'].sum().unstack()
cinsiyet_kategori.plot(kind='barh', ax=axes[0], color=['#FF6B6B', '#4ECDC4'], edgecolor='black', linewidth=0.5)
axes[0].set_title('Cinsiyete Göre Kategori Satışları', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Toplam Satış (₺)', fontsize=12)
axes[0].set_ylabel('')
axes[0].legend(title='Gender')

# Cinsiyete göre ortalama harcama
sns.boxplot(data=df, x='Gender', y='gross income', palette=['#FF6B6B', '#4ECDC4'], ax=axes[1])
axes[1].set_title('Cinsiyete Göre Harcama Dağılımı', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender', fontsize=12)
axes[1].set_ylabel('Brüt Gelir (₺)', fontsize=12)

plt.tight_layout()
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Cinsiyet ve ödeme yöntemi
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Payment', hue='Gender', palette=['#FF6B6B', '#4ECDC4'])
plt.title('Cinsiyete Göre Ödeme Yöntemi Tercihi', fontsize=16, fontweight='bold')
plt.xlabel('Payment', fontsize=12)
plt.ylabel('İşlem Sayısı', fontsize=12)
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

## 6. Şube Karşılaştırma Kutu Grafiği

### Kutu Grafiği (Box Plot)

Kutu grafiği verinin dağılımını, medyanını, çeyrekliklerini ve uç değerlerini (outlier) gösterir. Gruplar arası karşılaştırma için idealdir.

In [ ]:
# Şubeye göre brüt gelir kutu grafiği
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Kutu grafiği
sns.boxplot(data=df, x='Branch', y='gross income', palette='Set2', ax=axes[0])
axes[0].set_title('Şubelere Göre Satış Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Branch', fontsize=12)
axes[0].set_ylabel('Brüt Gelir (₺)', fontsize=12)

# Şube ve kategori bazlı toplam
sube_kategori = df.groupby(['Branch', 'Product line'])['gross income'].sum().unstack()
sube_kategori.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set3', edgecolor='black', linewidth=0.3)
axes[1].set_title('Şube ve Kategoriye Göre Toplam Satış', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Branch', fontsize=12)
axes[1].set_ylabel('Toplam Satış (₺)', fontsize=12)
axes[1].legend(title='Product line', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Şube bazında müşteri puanı dağılımı
plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x='Branch', y='Rating', palette='muted', inner='box')
plt.title('Şubelere Göre Müşteri Memnuniyet Puanı', fontsize=16, fontweight='bold')
plt.xlabel('Branch', fontsize=12)
plt.ylabel('Rating', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Korelasyon Isı Haritası

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Sayısal sütunların korelasyonu
sayisal = df[['Unit price', 'Quantity', 'Total', 'Tax 5%', 'gross income', 'Rating']]
korelasyon = sayisal.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(korelasyon, dtype=bool))
sns.heatmap(korelasyon, annot=True, cmap='RdYlBu_r', center=0,
            fmt='.3f', linewidths=2, linecolor='white',
            mask=mask, square=True, cbar_kws={'shrink': 0.8},
            annot_kws={'fontsize': 12, 'fontweight': 'bold'})
plt.title('Süpermarket Verisi - Korelasyon Matrisi', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Bonus: Şube ve gün bazında ortalama satış ısı haritası
df['Gün'] = df['Date'].dt.day_name()
gun_sirasi = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
gun_tr = {'Monday': 'Pazartesi', 'Tuesday': 'Salı', 'Wednesday': 'Çarşamba',
           'Thursday': 'Perşembe', 'Friday': 'Cuma', 'Saturday': 'Cumartesi', 'Sunday': 'Pazar'}

gun_sube = df.pivot_table(values='gross income', index='Gün', columns='Branch', aggfunc='mean')
gun_sube = gun_sube.reindex(gun_sirasi)
gun_sube.index = [gun_tr[g] for g in gun_sube.index]

plt.figure(figsize=(10, 7))
sns.heatmap(gun_sube, annot=True, cmap='YlOrRd', fmt='.0f',
            linewidths=2, linecolor='white', cbar_kws={'label': 'Ortalama Satış (₺)'})
plt.title('Gün ve Şubeye Göre Ortalama Satış', fontsize=16, fontweight='bold')
plt.xlabel('Branch', fontsize=12)
plt.ylabel('Gün', fontsize=12)
plt.tight_layout()
plt.show()

---

## Özet

Bu defterde süpermarket satış verisi üzerinde şu analizleri gerçekleştirdik:

| Analiz | Grafik Türü | Bulgular |
|--------|-------------|----------|
| Ürün kategorisi satışları | Yatay çubuk | En çok satan kategori belirlendi |
| Aylık satış trendi | Çizgi + hareketli ortalama | Zaman içindeki satış değişimi |
| Ödeme yöntemi | Pasta grafiği | Adet ve tutar bazlı dağılım |
| Cinsiyet analizi | Çubuk + kutu grafiği | Cinsiyet bazlı alışveriş farkları |
| Şube karşılaştırma | Kutu + yığılmış çubuk | Şubeler arası performans farkı |
| Korelasyon | Isı haritası | Değişkenler arası ilişkiler |

Bir sonraki defterde Google Trends verilerini analiz edeceğiz!

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>